In [10]:
%load_ext autoreload
%autoreload 2

In [11]:
import os
os.chdir(r"C:\Users\cesai\Projects\var_backtest")

In [12]:
from src.db import get_connection
from src.ingest import download_prices
from src.ingest import load_raw_to_db

In [13]:
df = download_prices(tickers=['^GSPC'], start='2026-01-01', end='2026-04-15')

[*********************100%***********************]  1 of 1 completed


In [16]:
import numpy as np

df['log_return'] = np.log(df['adj_close'] / df['adj_close'].shift(-1))

In [20]:
from src.models.hs import hs_var 
hs_result = hs_var(df['log_return'], window=20, confidence=0.9)


In [ ]:
from src.models.ewma import ewma_var


ewma_result = ewma_var(df['log_return'], confidence=0.99)

print(ewma_result.shape, df['log_return'].shape)   
print(ewma_result.isna().sum())         
ewma_result.dropna().head()

(70,) (70,)
1


1   -0.002619
2   -0.016507
3   -0.013279
4   -0.022875
5   -0.022451
Name: log_return, dtype: float64

In [26]:
from src.models.garch import garch_var

garch_result = garch_var(df['log_return'], window=20, confidence=0.99)

print(garch_result.shape, df['log_return'].shape)   
print(garch_result.isna().sum())         
garch_result.dropna().head()

(70,) (70,)
20


20   -0.029531
21   -0.028947
22   -0.030061
23   -0.025923
24   -0.022726
Name: log_return, dtype: float64

In [ ]:
import matplotlib.pyplot as plt

returns = df['log_return']

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(returns.index, returns, label='实际收益率', alpha=0.5, lw=0.7)
ax.plot(hs_result.index, -hs_result, label='HS -VaR(99%)', color='red')
ax.plot(ewma_result.index, -ewma_result, label='EWMA -VaR(99%)', color='orange')
ax.legend()
ax.set_title('VaR 预测 vs 实际收益率')
plt.show()

AttributeError: module 'matplotlib' has no attribute 'subplots'